# Build pipeline

End-to-end FinQA preparation: **download** raw FinQA from GitHub, **filter** for mathematical reasoning, **preprocess** to instruction/input/response format, and **validate** training quality.

Run cells in order (or *Run All*). The pipeline order is: **downloader** → **filter** → **preprocessor** → **validator**.


## 1. Dataset downloader (Phase 2)


#### Imports and Styling Utilities

In [16]:
import requests
from zipfile import ZipFile
import tempfile
import json
import os
import html
from tqdm.notebook import tqdm # Using .notebook for better Jupyter bars
from IPython.display import HTML, display

def _dinfo(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;">{html.escape(str(msg))}</p>'))

def _dwarn(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;color:#a16207;">{html.escape(str(msg))}</p>'))

def _derr(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;color:#b91c1c;font-weight:500;">{html.escape(str(msg))}</p>'))

#### The Download Logic

In [17]:
def download_finqa():
    """Download FinQA dataset from original GitHub source"""
    _dinfo("Starting FinQA dataset download from original GitHub source...")
    _dinfo("Source: https://github.com/czyssrs/FinQA")

    # GitHub repository ZIP URL
    github_zip_url = "https://github.com/czyssrs/FinQA/archive/refs/heads/main.zip"

    try:
        # Create directory if it doesn't exist
        os.makedirs("raw_data", exist_ok=True)

        _dinfo("Downloading FinQA repository...")

        # Download the ZIP file
        response = requests.get(github_zip_url, stream=True)
        response.raise_for_status()

        total_size = int(response.headers.get('content-length', 0))

        with tempfile.NamedTemporaryFile(delete=False, suffix='.zip') as tmp_file:
            # Download with progress bar
            with tqdm(total=total_size, unit='iB', unit_scale=True, desc="Downloading") as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        tmp_file.write(chunk)
                        pbar.update(len(chunk))

            tmp_zip_path = tmp_file.name

        _dinfo("Extracting dataset files...")

        # Extract and process JSON files
        total_examples = 0

        with ZipFile(tmp_zip_path, 'r') as zip_ref:
            # File mappings: original -> our naming convention
            file_mappings = {
                'FinQA-main/dataset/train.json': 'raw_data/finqa_train.json',
                'FinQA-main/dataset/dev.json': 'raw_data/finqa_validation.json',
                'FinQA-main/dataset/test.json': 'raw_data/finqa_test.json'
            }

            for original_path, output_file in file_mappings.items():
                try:
                    _dinfo(f"Processing {original_path}...")

                    # Extract and read the JSON file
                    with zip_ref.open(original_path) as json_file:
                        data = json.load(json_file)

                    # Validate structure
                    if not data:
                        _dwarn(f"Empty data in {original_path}")
                        continue

                    # Check if data contains expected fields
                    sample_item = data[0] if isinstance(data, list) else data
                    if isinstance(sample_item, dict) and 'qa' in sample_item:
                        # Extract qa fields to top level for easier processing
                        processed_data = []

                        for item in tqdm(data, desc=f"Processing {os.path.basename(output_file)}"):
                            if 'qa' in item:
                                qa_data = item['qa']
                                processed_item = {
                                    'id': item.get('id', ''),
                                    'pre_text': item.get('pre_text', []),
                                    'post_text': item.get('post_text', []),
                                    'table': item.get('table', []),
                                    'question': qa_data.get('question', ''),
                                    'answer': qa_data.get('answer', ''),
                                    'program': qa_data.get('program', []),
                                    'program_re': qa_data.get('program_re', ''),
                                    'gold_inds': qa_data.get('gold_inds', {})
                                }
                                processed_data.append(processed_item)

                        data = processed_data

                    # Save processed data
                    with open(output_file, 'w', encoding='utf-8') as f:
                        json.dump(data, f, indent=2, ensure_ascii=False)

                    _dinfo(f"✅ Saved {len(data)} examples to {output_file}")
                    total_examples += len(data)

                    # Show sample fields for verification
                    if data and isinstance(data[0], dict):
                        sample_keys = list(data[0].keys())
                        display("Sample field names")
                        display(sample_keys)
                        # Check for program fields specifically
                        has_program = any(key in sample_keys for key in ['program', 'program_re'])
                        if has_program:
                            _dinfo(f"  ✅ Contains reasoning programs!")
                        else:
                            _dwarn(f"  ⚠️ No program fields found")

                except KeyError as e:
                    _derr(f"File {original_path} not found in archive: {e}")
                except Exception as e:
                    _derr(f"Error processing {original_path}: {e}")

        # Clean up temporary file
        os.unlink(tmp_zip_path)

        if total_examples > 0:
            _dinfo(f"✅ FinQA download complete! Total examples: {total_examples}")
            _dinfo("Original FinQA data includes reasoning programs (program/program_re fields)")
            return True
        else:
            _derr("❌ No data was successfully downloaded")
            return False

    except Exception as e:
        _derr(f"Error downloading FinQA from GitHub: {str(e)}")
        return False


#### Execution

In [20]:
_dinfo("=== PHASE 2: DATASET DOWNLOADER ===")

# Step 1: Execute Download
finqa_success = download_finqa()

# Step 2: Verify
if finqa_success:
    _dinfo("✅ FinQA dataset ready for analysis!")
else:
    _derr("❌ Download failed.")

Downloading: 0.00iB [00:00, ?iB/s]

Processing finqa_train.json:   0%|          | 0/6251 [00:00<?, ?it/s]

'Sample field names'

['id',
 'pre_text',
 'post_text',
 'table',
 'question',
 'answer',
 'program',
 'program_re',
 'gold_inds']

Processing finqa_validation.json:   0%|          | 0/883 [00:00<?, ?it/s]

'Sample field names'

['id',
 'pre_text',
 'post_text',
 'table',
 'question',
 'answer',
 'program',
 'program_re',
 'gold_inds']

Processing finqa_test.json:   0%|          | 0/1147 [00:00<?, ?it/s]

'Sample field names'

['id',
 'pre_text',
 'post_text',
 'table',
 'question',
 'answer',
 'program',
 'program_re',
 'gold_inds']

## 2. Data filter


In [ ]:
"""
Phase 2: Data Filter
Filters FinQA dataset to keep only examples with mathematical reasoning.
This is the core algorithm that ensures quality training data for financial calculations.
"""

import json
import re
import os
from typing import List, Dict, Any, Tuple
from tqdm import tqdm


import html
from IPython.display import display, HTML, Markdown, JSON


def _dinfo(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;">{html.escape(str(msg))}</p>'))


def _dwarn(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;color:#a16207;">{html.escape(str(msg))}</p>'))


def _derr(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;color:#b91c1c;font-weight:500;">{html.escape(str(msg))}</p>'))



def has_valid_program(item: Dict[Any, Any]) -> Tuple[bool, str]:
    """
    Check if an item has a valid program_re (mathematical reasoning steps)
    Returns (is_valid, reason)
    """
    program = item.get('program_re', '') or item.get('program', '')

    if not program or len(program.strip()) == 0:
        return False, "No program found"

    # Convert to string if it's a list
    if isinstance(program, list):
        program = '\n'.join(str(p) for p in program)

    program_str = str(program).lower()

    # Check for mathematical operations - these indicate computational reasoning
    math_patterns = [
        r'divide\s*\(',
        r'multiply\s*\(',
        r'add\s*\(',
        r'subtract\s*\(',
        r'greater\s*\(',
        r'less\s*\(',
        r'exp\s*\(',
        r'table_\w+\(',
        r'#\d+\s*=',  # Variable assignments like #0 =
        r'\+|\-|\*|\/|\%',  # Basic math operators
    ]

    has_math = any(re.search(pattern, program_str, re.IGNORECASE) for pattern in math_patterns)

    if not has_math:
        return False, "No mathematical operations found"

    # Check for multistep reasoning (more than just a lookup)
    reasoning_indicators = [
        len(program.split('\n')) > 1,  # Multiple lines
        'divide(' in program_str and ('multiply(' in program_str or 'add(' in program_str),  # Multiple operations
        program_str.count('=') > 1,  # Multiple variable assignments
        'table_' in program_str and any(op in program_str for op in ['divide', 'multiply', 'add', 'subtract'])
    ]

    has_reasoning = any(reasoning_indicators)

    if not has_reasoning:
        return False, "Simple lookup, no multi-step reasoning"

    # Avoid examples that are too complex or malformed
    if len(program.split('\n')) > 10:
        return False, "Program too complex (>10 steps)"

    return True, "Valid mathematical reasoning"

def extract_reasoning_steps(program_re: str, question: str = "") -> List[str]:
    """
    Parse program_re into human-readable reasoning steps
    """
    if not program_re:
        return []

    # Convert to string if it's a list
    if isinstance(program_re, list):
        program_re = '\n'.join(str(p) for p in program_re)

    steps = []
    lines = str(program_re).strip().split('\n')

    for i, line in enumerate(lines, 1):
        line = line.strip()
        if not line:
            continue

        # Convert program operations to natural language
        step_text = ""

        if 'table_' in line.lower():
            if 'max(' in line.lower():
                step_text = f"Step {i}: Find the maximum value from the financial table"
            elif 'min(' in line.lower():
                step_text = f"Step {i}: Find the minimum value from the financial table"
            elif 'sum(' in line.lower():
                step_text = f"Step {i}: Sum values from the financial table"
            else:
                step_text = f"Step {i}: Extract data from the financial table"

        elif 'divide(' in line.lower():
            step_text = f"Step {i}: Perform division calculation"
        elif 'multiply(' in line.lower():
            step_text = f"Step {i}: Perform multiplication calculation"
        elif 'add(' in line.lower():
            step_text = f"Step {i}: Perform addition calculation"
        elif 'subtract(' in line.lower():
            step_text = f"Step {i}: Perform subtraction calculation"
        elif 'greater(' in line.lower():
            step_text = f"Step {i}: Compare values (greater than)"
        elif 'less(' in line.lower():
            step_text = f"Step {i}: Compare values (less than)"
        elif '#' in line and '=' in line:
            step_text = f"Step {i}: Store intermediate calculation result"
        else:
            # Keep original line but make it more readable
            clean_line = line.replace('(', ' (').replace(')', ') ').replace(',', ', ')
            step_text = f"Step {i}: {clean_line}"

        steps.append(step_text)

    return steps

def analyze_dataset_structure(filepath: str) -> Dict[str, Any]:
    """
    Analyze the structure of a dataset file to understand its format
    """
    _dinfo(f"Analyzing structure of {filepath}")

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)

        if not data:
            return {"error": "Empty dataset"}

        # Analyze first few items
        sample_size = min(5, len(data))
        sample_items = data[:sample_size]

        # Get all unique keys
        all_keys = set()
        for item in sample_items:
            if isinstance(item, dict):
                all_keys.update(item.keys())

        # Check for program fields
        program_fields = [key for key in all_keys if 'program' in key.lower()]

        # Sample values for key fields
        key_samples = {}
        important_keys = ['question', 'answer'] + program_fields + ['table', 'text']

        for key in important_keys:
            if key in all_keys:
                values = [item.get(key) for item in sample_items if item.get(key)]
                if values:
                    key_samples[key] = values[0]

        return {
            "total_items": len(data),
            "all_keys": sorted(list(all_keys)),
            "program_fields": program_fields,
            "key_samples": key_samples
        }

    except Exception as e:
        return {"error": str(e)}

def filter_financial_reasoning_data(input_file: str, output_file: str) -> int:
    """
    Filter dataset to keep only examples with mathematical reasoning
    """
    _dinfo(f"Filtering {input_file}...")

    # First analyze the dataset structure
    structure = analyze_dataset_structure(input_file)
    if "error" in structure:
        _derr(f"Error analyzing {input_file}: {structure['error']}")
        return 0

    _dinfo(f"Dataset structure: {len(structure['all_keys'])} fields, {structure['total_items']} total items")
    if structure['program_fields']:
        _dinfo(f"Program fields found: {structure['program_fields']}")
    else:
        _dwarn("No program fields found - this dataset may not contain reasoning steps")

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        _derr(f"Error reading {input_file}: {str(e)}")
        return 0

    filtered_data = []
    filter_stats = {
        "total": len(data),
        "no_program": 0,
        "no_math": 0,
        "simple_lookup": 0,
        "too_complex": 0,
        "valid": 0
    }

    for item in tqdm(data, desc="Filtering examples"):
        is_valid, reason = has_valid_program(item)

        # Update statistics
        if "No program found" in reason:
            filter_stats["no_program"] += 1
        elif "No mathematical operations found" in reason:
            filter_stats["no_math"] += 1
        elif "Simple lookup" in reason:
            filter_stats["simple_lookup"] += 1
        elif "too complex" in reason:
            filter_stats["too_complex"] += 1
        elif is_valid:
            filter_stats["valid"] += 1

        if is_valid:
            # Get the program field (try different possible names)
            program = item.get('program_re', '') or item.get('program', '') or item.get('reasoning', '')

            # Create enhanced item with reasoning steps
            enhanced_item = {
                'question': item.get('question', ''),
                'answer': item.get('answer', ''),
                'program_re': program,
                'reasoning_steps': extract_reasoning_steps(program, item.get('question', '')),
                'table': item.get('table', ''),
                'text': item.get('text', ''),
                'source': 'finqa',
                'original_keys': list(item.keys())  # Keep track of original structure
            }

            # Add any additional context fields that might be useful
            for key in ['context', 'gold_inds', 'question_type']:
                if key in item:
                    enhanced_item[key] = item[key]

            filtered_data.append(enhanced_item)

    # Save filtered data
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(filtered_data, f, indent=2, ensure_ascii=False)

    # Display detailed statistics
    display(Markdown(f"### Filtering results — `{os.path.basename(input_file)}`"))
    _ts = filter_stats["total"]

    def _fs_pct(c: int) -> float:
        return round(100.0 * c / _ts, 1) if _ts else 0.0

    display(
        JSON(
            {
                "total": _ts,
                "no_program": {"count": filter_stats["no_program"], "pct": _fs_pct(filter_stats["no_program"])},
                "no_math": {"count": filter_stats["no_math"], "pct": _fs_pct(filter_stats["no_math"])},
                "simple_lookup": {"count": filter_stats["simple_lookup"], "pct": _fs_pct(filter_stats["simple_lookup"])},
                "too_complex": {"count": filter_stats["too_complex"], "pct": _fs_pct(filter_stats["too_complex"])},
                "valid": {"count": filter_stats["valid"], "pct": _fs_pct(filter_stats["valid"])},
            }
        )
    )
    _dinfo(f"Saved to: {output_file}")

    return len(filtered_data)

def main():
    """Filter FinQA dataset"""
    _dinfo("=== PHASE 2: DATA FILTERING ===")

    os.makedirs("filtered_data", exist_ok=True)

    total_filtered = 0
    datasets_processed = 0

    # Process all JSON files in raw_data directory
    raw_data_dir = "raw_data"

    if not os.path.exists(raw_data_dir):
        _derr(f"Raw data directory {raw_data_dir} not found!")
        _dinfo("Run the dataset downloader cell first so raw_data/ exists")
        return False

    json_files = [f for f in os.listdir(raw_data_dir) if f.endswith('.json')]

    if not json_files:
        _derr("No JSON files found in raw_data directory!")
        return False

    _dinfo(f"Found {len(json_files)} dataset files to process")

    for filename in sorted(json_files):
        input_file = os.path.join(raw_data_dir, filename)

        # Create output filename
        base_name = filename.replace('.json', '')
        output_file = f"filtered_data/{base_name}_filtered.json"

        _dinfo(f"\nProcessing: {filename}")

        try:
            count = filter_financial_reasoning_data(input_file, output_file)
            total_filtered += count
            datasets_processed += 1

            _dinfo(f"✅ Successfully filtered {filename}: {count} examples")

        except Exception as e:
            _derr(f"❌ Error processing {filename}: {str(e)}")

    # Final summary
    _dinfo(f"\n=== FILTERING COMPLETE ===")
    _dinfo(f"Datasets processed: {datasets_processed}/{len(json_files)}")
    _dinfo(f"Total examples with mathematical reasoning: {total_filtered}")

    if total_filtered > 0:
        _dinfo("✅ Data filtering successful!")
        _dinfo("Next step: run the data preprocessor cell to convert to training format")
        return True
    else:
        _derr("❌ No valid examples found!")
        return False

if __name__ == "__main__":
    main()


## 3. Data preprocessor


In [ ]:
"""
Phase 2: Data Preprocessor
Converts filtered financial reasoning data to instruction/input/response format for model training.
"""

import json
import os
from typing import List, Dict, Any
from tqdm import tqdm
import random


import html
from IPython.display import display, HTML, Markdown, JSON


def _dinfo(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;">{html.escape(str(msg))}</p>'))


def _dwarn(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;color:#a16207;">{html.escape(str(msg))}</p>'))


def _derr(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;color:#b91c1c;font-weight:500;">{html.escape(str(msg))}</p>'))



def format_table_data(table_data: Any) -> str:
    """
    Convert table data to a readable string format
    """
    if not table_data:
        return ""

    # Handle different table formats
    if isinstance(table_data, str):
        # Already a string, just clean it up
        return table_data.strip()

    elif isinstance(table_data, dict):
        # Dictionary format - convert to readable table
        if 'header' in table_data and 'rows' in table_data:
            # Structured table format
            header = table_data.get('header', [])
            rows = table_data.get('rows', [])

            if not header or not rows:
                return str(table_data)

            # Create table string
            table_str = " | ".join(str(h) for h in header) + "\n"
            table_str += " | ".join(["-" * len(str(h)) for h in header]) + "\n"

            for row in rows[:10]:  # Limit to 10 rows for readability
                if isinstance(row, list):
                    table_str += " | ".join(str(cell) for cell in row) + "\n"
                else:
                    table_str += str(row) + "\n"

            if len(rows) > 10:
                table_str += f"... ({len(rows) - 10} more rows)\n"

            return table_str.strip()
        else:
            # Generic dictionary - convert to key-value pairs
            items = []
            for key, value in table_data.items():
                if isinstance(value, (list, dict)):
                    value = str(value)[:100] + "..." if len(str(value)) > 100 else str(value)
                items.append(f"{key}: {value}")
            return "\n".join(items)

    elif isinstance(table_data, list):
        # List format
        if not table_data:
            return ""

        # Check if it's a list of lists (rows)
        if isinstance(table_data[0], list):
            table_str = ""
            for i, row in enumerate(table_data[:10]):
                table_str += " | ".join(str(cell) for cell in row) + "\n"

            if len(table_data) > 10:
                table_str += f"... ({len(table_data) - 10} more rows)\n"

            return table_str.strip()
        else:
            # List of values
            return " | ".join(str(item) for item in table_data[:20])

    else:
        return str(table_data)

def create_instruction_format(item: Dict, instruction_type: str = "standard") -> Dict:
    """
    Convert filtered data to Instruction/Input/Response format
    """
    # Build context from table and text
    context_parts = []

    # Add table data if available
    table_content = format_table_data(item.get('table'))
    if table_content:
        context_parts.append(f"Financial Data Table:\n{table_content}")

    # Add text context if available
    text_content = item.get('text', '').strip()
    if text_content:
        # Limit text length to avoid overly long inputs
        if len(text_content) > 1000:
            text_content = text_content[:1000] + "... [truncated]"
        context_parts.append(f"Additional Context:\n{text_content}")

    context = "\n\n".join(context_parts)

    # Create different instruction variations for diversity
    instructions = {
        "standard": """You are a financial analyst AI. Given the financial data and question below, provide step-by-step reasoning to calculate the answer. Show your mathematical steps clearly and explain your reasoning.""",

        "detailed": """You are an expert financial analyst. Analyze the provided financial data and answer the question with detailed step-by-step calculations. Break down each mathematical operation and explain the financial reasoning behind each step.""",

        "concise": """Analyze the financial data and answer the question. Show the calculation steps and provide the final numerical answer.""",

        "educational": """As a financial tutor, help solve this problem by showing clear step-by-step calculations. Explain each step so that someone learning financial analysis can understand the process."""
    }

    # Select instruction based on type
    instruction = instructions.get(instruction_type, instructions["standard"])

    # Create the input
    input_parts = [f"Question: {item['question']}"]
    if context:
        input_parts.append(context)

    input_text = "\n\n".join(input_parts)

    # Create the response with reasoning steps
    reasoning_steps = item.get('reasoning_steps', [])
    if reasoning_steps:
        reasoning_text = "\n".join(reasoning_steps)
    else:
        # Fallback to program_re if no reasoning steps
        program = item.get('program_re', '')
        reasoning_text = f"Calculation steps:\n{program}"

    # Format the final answer
    answer = item.get('answer', '')
    if isinstance(answer, (int, float)):
        answer_text = f"Final Answer: {answer}"
    elif isinstance(answer, str) and answer.strip():
        answer_text = f"Final Answer: {answer.strip()}"
    else:
        answer_text = "Final Answer: [See calculation above]"

    response = f"{reasoning_text}\n\n{answer_text}"

    return {
        "instruction": instruction,
        "input": input_text,
        "output": response,
        "source": item.get('source', 'unknown'),
        "question_length": len(item.get('question', '')),
        "has_table": bool(item.get('table')),
        "has_text": bool(item.get('text', '').strip()),
        "reasoning_steps_count": len(reasoning_steps)
    }

def validate_training_example(example: Dict) -> bool:
    """
    Validate that a training example meets quality standards
    """
    # Check required fields
    required_fields = ['instruction', 'input', 'output']
    if not all(field in example and example[field].strip() for field in required_fields):
        return False

    # Check minimum lengths
    if len(example['input']) < 20:  # Too short input
        return False

    if len(example['output']) < 30:  # Too short output
        return False

    # Check for mathematical content in output
    math_indicators = ['step', 'calculate', 'divide', 'multiply', 'add', 'subtract', 'answer']
    has_math = any(indicator in example['output'].lower() for indicator in math_indicators)

    if not has_math:
        return False

    return True

def preprocess_for_training(
    input_dir: str = "filtered_data",
    output_file: str = "training/finqa_clean.json",
    max_examples: int = None,
    instruction_variety: bool = True
) -> int:
    """
    Convert all filtered data to training format
    """
    _dinfo("Converting filtered data to training format...")

    if not os.path.exists(input_dir):
        _derr(f"Input directory {input_dir} not found!")
        return 0

    training_data = []
    file_stats = {}

    # Get all filtered files
    filtered_files = [f for f in os.listdir(input_dir) if f.endswith('_filtered.json')]

    if not filtered_files:
        _derr(f"No filtered JSON files found in {input_dir}")
        _dinfo("Run the data filter cell first")
        return 0

    _dinfo(f"Processing {len(filtered_files)} filtered files...")

    # Process each filtered file
    for filename in sorted(filtered_files):
        filepath = os.path.join(input_dir, filename)
        _dinfo(f"Processing {filename}...")

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)

            file_examples = 0

            for item in tqdm(data, desc=f"Converting {filename}"):
                # Randomly vary instruction type for diversity
                if instruction_variety:
                    instruction_types = ["standard", "detailed", "concise", "educational"]
                    instruction_type = random.choice(instruction_types)
                else:
                    instruction_type = "standard"

                try:
                    formatted_item = create_instruction_format(item, instruction_type)

                    # Validate the example
                    if validate_training_example(formatted_item):
                        training_data.append(formatted_item)
                        file_examples += 1

                        # Check if we've reached max examples
                        if max_examples and len(training_data) >= max_examples:
                            _dinfo(f"Reached maximum examples limit: {max_examples}")
                            break

                except Exception as e:
                        _dwarn(f"Error processing item: {str(e)}")
                        continue

            file_stats[filename] = file_examples
            _dinfo(f"Converted {file_examples} examples from {filename}")

            # Break if we've reached max examples
            if max_examples and len(training_data) >= max_examples:
                break

        except Exception as e:
            _derr(f"Error processing {filename}: {str(e)}")
            continue

    if not training_data:
        _derr("No valid training examples created!")
        return 0

    # Shuffle the training data for better training
    random.shuffle(training_data)

    # Save training data
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(training_data, f, indent=2, ensure_ascii=False)

    # Create summary statistics
    create_preprocessing_summary(training_data, file_stats, output_file)

    _dinfo(f"✅ Created {len(training_data)} training examples")
    _dinfo(f"📁 Saved to {output_file}")

    return len(training_data)

def create_preprocessing_summary(training_data: List[Dict], file_stats: Dict, output_file: str):
    """
    Create a summary of the preprocessing results
    """
    summary = {
        "total_examples": len(training_data),
        "source_breakdown": {},
        "quality_metrics": {
            "avg_input_length": 0,
            "avg_output_length": 0,
            "examples_with_tables": 0,
            "examples_with_text": 0,
            "avg_reasoning_steps": 0
        },
        "file_breakdown": file_stats
    }

    # Calculate metrics
    if training_data:
        # Source breakdown
        for item in training_data:
            source = item.get('source', 'unknown')
            summary["source_breakdown"][source] = summary["source_breakdown"].get(source, 0) + 1

        # Quality metrics
        total_input_len = sum(len(item['input']) for item in training_data)
        total_output_len = sum(len(item['output']) for item in training_data)

        summary["quality_metrics"]["avg_input_length"] = int(total_input_len / len(training_data))
        summary["quality_metrics"]["avg_output_length"] = int(total_output_len / len(training_data))
        summary["quality_metrics"]["examples_with_tables"] = sum(1 for item in training_data if item.get('has_table', False))
        summary["quality_metrics"]["examples_with_text"] = sum(1 for item in training_data if item.get('has_text', False))

        reasoning_steps = [item.get('reasoning_steps_count', 0) for item in training_data]
        if reasoning_steps:
            summary["quality_metrics"]["avg_reasoning_steps"] = sum(reasoning_steps) / len(reasoning_steps)

    # Save summary
    summary_file = output_file.replace('.json', '_summary.json')
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    # Display summary
    display(Markdown("### Preprocessing summary"))
    display(JSON(summary))
    _dinfo(f"Summary saved to: {summary_file}")

def main():
    """Main preprocessing function"""
    _dinfo("=== PHASE 2: DATA PREPROCESSING ===")

    # Allow customization via command line or defaults
    max_examples = None  # Set to a number like 10000 to limit dataset size

    examples_created = preprocess_for_training(
        input_dir="filtered_data",
        output_file="training/finqa_clean.json",
        max_examples=max_examples,
        instruction_variety=True
    )

    if examples_created > 0:
        _dinfo("✅ Preprocessing complete!")
        _dinfo("Next step: run the data validator cell to validate training data quality")

        # Show sample example
        try:
            with open("training/finqa_clean.json", 'r', encoding='utf-8') as f:
                data = json.load(f)

            if data:
                display(Markdown("### Sample training example (first record)"))
                sample = data[0]

                def _clip(s: str, n: int = 500) -> str:
                    s = s or ""
                    return s if len(s) <= n else s[:n] + "..."

                display(
                    JSON(
                        {
                            "instruction": _clip(sample.get("instruction", ""), 600),
                            "input": _clip(sample.get("input", ""), 800),
                            "output": _clip(sample.get("output", ""), 800),
                            "source": sample.get("source", "unknown"),
                        }
                    )
                )

        except Exception as e:
            _dwarn(f"Could not load sample: {str(e)}")

        return True
    else:
        _derr("❌ Preprocessing failed!")
        return False

if __name__ == "__main__":
    main()


## 4. Data validator


In [ ]:
"""
Phase 2: Data Validator
Validates the quality and completeness of training data for financial reasoning model.
"""

import json
import re
import os
from collections import Counter, defaultdict
from typing import Dict, List, Any, Tuple
from tqdm import tqdm

import html
from IPython.display import display, HTML, Markdown, JSON


def _dinfo(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;">{html.escape(str(msg))}</p>'))


def _dwarn(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;color:#a16207;">{html.escape(str(msg))}</p>'))


def _derr(msg) -> None:
    display(HTML(f'<p style="margin:0.2em 0;color:#b91c1c;font-weight:500;">{html.escape(str(msg))}</p>'))



def validate_training_data(data_file: str = "training/finqa_clean.json") -> Dict[str, Any]:
    """
    Comprehensive validation of training data quality
    """
    _dinfo(f"Validating training data: {data_file}")

    if not os.path.exists(data_file):
        _derr(f"Training data file not found: {data_file}")
        return {"error": f"File not found: {data_file}"}

    try:
        with open(data_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        _derr(f"Error loading training data: {str(e)}")
        return {"error": f"Failed to load data: {str(e)}"}

    if not data:
        _derr("Training data is empty!")
        return {"error": "Empty dataset"}

    _dinfo(f"Loaded {len(data)} training examples")

    validation_results = {
        "total_examples": len(data),
        "field_completeness": {},
        "content_quality": {},
        "reasoning_analysis": {},
        "mathematical_operations": {},
        "length_statistics": {},
        "source_distribution": {},
        "quality_issues": [],
        "recommendations": []
    }

    # Validate field completeness
    validation_results["field_completeness"] = validate_field_completeness(data)

    # Analyze content quality
    validation_results["content_quality"] = analyze_content_quality(data)

    # Analyze reasoning patterns
    validation_results["reasoning_analysis"] = analyze_reasoning_patterns(data)

    # Check mathematical operations coverage
    validation_results["mathematical_operations"] = analyze_mathematical_operations(data)

    # Analyze length statistics
    validation_results["length_statistics"] = analyze_length_statistics(data)

    # Check source distribution
    validation_results["source_distribution"] = analyze_source_distribution(data)

    # Identify quality issues and recommendations
    validation_results["quality_issues"] = identify_quality_issues(data, validation_results)
    validation_results["recommendations"] = generate_recommendations(validation_results)

    return validation_results

def validate_field_completeness(data: List[Dict]) -> Dict[str, Any]:
    """Check completeness of required fields"""
    _dinfo("Validating field completeness...")

    required_fields = ['instruction', 'input', 'output']
    optional_fields = ['source', 'has_table', 'has_text', 'reasoning_steps_count']

    field_stats = {}

    for field in required_fields + optional_fields:
        present_count = sum(1 for item in data if field in item and item[field])
        field_stats[field] = {
            "present": present_count,
            "missing": len(data) - present_count,
            "percentage": (present_count / len(data)) * 100
        }

    return field_stats

def analyze_content_quality(data: List[Dict]) -> Dict[str, Any]:
    """Analyze the quality of content in each field"""
    _dinfo("Analyzing content quality...")

    quality_metrics = {
        "empty_instructions": 0,
        "empty_inputs": 0,
        "empty_outputs": 0,
        "very_short_inputs": 0,  # < 50 chars
        "very_short_outputs": 0,  # < 100 chars
        "very_long_inputs": 0,  # > 2000 chars
        "very_long_outputs": 0,  # > 1000 chars
        "malformed_examples": 0
    }

    for item in data:
        instruction = item.get('instruction', '').strip()
        input_text = item.get('input', '').strip()
        output_text = item.get('output', '').strip()

        # Check for empty fields
        if not instruction:
            quality_metrics["empty_instructions"] += 1
        if not input_text:
            quality_metrics["empty_inputs"] += 1
        if not output_text:
            quality_metrics["empty_outputs"] += 1

        # Check lengths
        if len(input_text) < 50:
            quality_metrics["very_short_inputs"] += 1
        if len(output_text) < 100:
            quality_metrics["very_short_outputs"] += 1
        if len(input_text) > 2000:
            quality_metrics["very_long_inputs"] += 1
        if len(output_text) > 1000:
            quality_metrics["very_long_outputs"] += 1

        # Check for malformed examples
        if not instruction or not input_text or not output_text:
            quality_metrics["malformed_examples"] += 1

    # Convert to percentages
    total = len(data)
    for key in quality_metrics:
        count = quality_metrics[key]
        quality_metrics[key] = {
            "count": count,
            "percentage": (count / total) * 100
        }

    return quality_metrics

def analyze_reasoning_patterns(data: List[Dict]) -> Dict[str, Any]:
    """Analyze patterns in reasoning steps"""
    _dinfo("Analyzing reasoning patterns...")

    step_patterns = []
    reasoning_quality = {
        "has_step_keywords": 0,
        "has_calculations": 0,
        "has_final_answer": 0,
        "multi_step_reasoning": 0
    }

    step_keywords = ['step', 'first', 'second', 'next', 'then', 'finally', 'calculate', 'divide', 'multiply']
    calculation_keywords = ['=', '+', '-', '*', '/', 'divide', 'multiply', 'add', 'subtract', 'calculate']
    answer_keywords = ['final answer', 'answer:', 'result:', 'solution:']

    for item in data:
        output = item.get('output', '').lower()

        # Count reasoning steps
        steps = len(re.findall(r'step \d+', output))
        step_patterns.append(steps)

        # Check quality indicators
        if any(keyword in output for keyword in step_keywords):
            reasoning_quality["has_step_keywords"] += 1

        if any(keyword in output for keyword in calculation_keywords):
            reasoning_quality["has_calculations"] += 1

        if any(keyword in output for keyword in answer_keywords):
            reasoning_quality["has_final_answer"] += 1

        if steps > 1 or len(output.split('\n')) > 3:
            reasoning_quality["multi_step_reasoning"] += 1

    return {
        "step_distribution": dict(Counter(step_patterns)),
        "avg_steps": sum(step_patterns) / len(step_patterns) if step_patterns else 0,
        "max_steps": max(step_patterns) if step_patterns else 0,
        "quality_indicators": {
            key: {
                "count": value,
                "percentage": (value / len(data)) * 100
            }
            for key, value in reasoning_quality.items()
        }
    }

def analyze_mathematical_operations(data: List[Dict]) -> Dict[str, Any]:
    """Analyze coverage of different mathematical operations"""
    _dinfo("Analyzing mathematical operations coverage...")

    operations = {
        'addition': ['add', '+', 'plus', 'sum'],
        'subtraction': ['subtract', '-', 'minus', 'difference'],
        'multiplication': ['multiply', '*', 'times', 'product'],
        'division': ['divide', '/', 'divided by', 'ratio'],
        'percentage': ['%', 'percent', 'percentage'],
        'comparison': ['greater', 'less', 'compare', 'higher', 'lower'],
        'average': ['average', 'mean'],
        'growth': ['growth', 'increase', 'decrease', 'change']
    }

    operation_counts = {op: 0 for op in operations.keys()}

    for item in data:
        output = item.get('output', '').lower()
        input_text = item.get('input', '').lower()
        combined_text = output + " " + input_text

        for operation, keywords in operations.items():
            if any(keyword in combined_text for keyword in keywords):
                operation_counts[operation] += 1

    # Convert to percentages
    total = len(data)
    operation_coverage = {}
    for operation, count in operation_counts.items():
        operation_coverage[operation] = {
            "count": count,
            "percentage": (count / total) * 100
        }

    return operation_coverage

def analyze_length_statistics(data: List[Dict]) -> Dict[str, Any]:
    """Analyze length statistics for inputs and outputs"""
    _dinfo("Analyzing length statistics...")

    input_lengths = [len(item.get('input', '')) for item in data]
    output_lengths = [len(item.get('output', '')) for item in data]

    def get_stats(lengths):
        if not lengths:
            return {}
        return {
            "min": min(lengths),
            "max": max(lengths),
            "avg": sum(lengths) / len(lengths),
            "median": sorted(lengths)[len(lengths)//2]
        }

    return {
        "input_lengths": get_stats(input_lengths),
        "output_lengths": get_stats(output_lengths),
        "length_distribution": {
            "very_short_inputs": sum(1 for l in input_lengths if l < 100),
            "short_inputs": sum(1 for l in input_lengths if 100 <= l < 300),
            "medium_inputs": sum(1 for l in input_lengths if 300 <= l < 800),
            "long_inputs": sum(1 for l in input_lengths if l >= 800),
            "very_short_outputs": sum(1 for l in output_lengths if l < 150),
            "short_outputs": sum(1 for l in output_lengths if 150 <= l < 400),
            "medium_outputs": sum(1 for l in output_lengths if 400 <= l < 800),
            "long_outputs": sum(1 for l in output_lengths if l >= 800),
        }
    }

def analyze_source_distribution(data: List[Dict]) -> Dict[str, Any]:
    """Analyze distribution of data sources"""
    _dinfo("Analyzing source distribution...")

    sources = [item.get('source', 'unknown') for item in data]
    source_counts = Counter(sources)

    total = len(data)
    source_distribution = {}

    for source, count in source_counts.items():
        source_distribution[source] = {
            "count": count,
            "percentage": (count / total) * 100
        }

    return source_distribution

def identify_quality_issues(data: List[Dict], validation_results: Dict) -> List[str]:
    """Identify potential quality issues in the dataset"""
    _dinfo("Identifying quality issues...")

    issues = []

    # Check field completeness issues
    field_stats = validation_results["field_completeness"]
    for field in ['instruction', 'input', 'output']:
        if field_stats[field]["percentage"] < 100:
            issues.append(f"Missing {field} in {field_stats[field]['missing']} examples ({100-field_stats[field]['percentage']:.1f}%)")

    # Check content quality issues
    quality = validation_results["content_quality"]
    if quality["very_short_outputs"]["percentage"] > 10:
        issues.append(f"Too many very short outputs ({quality['very_short_outputs']['percentage']:.1f}%)")

    if quality["malformed_examples"]["count"] > 0:
        issues.append(f"Found {quality['malformed_examples']['count']} malformed examples")

    # Check mathematical operations coverage
    math_ops = validation_results["mathematical_operations"]
    low_coverage_ops = [op for op, stats in math_ops.items() if stats["percentage"] < 20]
    if low_coverage_ops:
        issues.append(f"Low coverage for mathematical operations: {', '.join(low_coverage_ops)}")

    # Check reasoning quality
    reasoning = validation_results["reasoning_analysis"]["quality_indicators"]
    if reasoning["has_calculations"]["percentage"] < 80:
        issues.append(f"Only {reasoning['has_calculations']['percentage']:.1f}% of examples contain clear calculations")

    if reasoning["has_final_answer"]["percentage"] < 90:
        issues.append(f"Only {reasoning['has_final_answer']['percentage']:.1f}% of examples have clear final answers")

    return issues

def generate_recommendations(validation_results: Dict) -> List[str]:
    """Generate recommendations based on validation results"""
    _dinfo("Generating recommendations...")

    recommendations = []

    total_examples = validation_results["total_examples"]

    # Dataset size recommendations
    if total_examples < 1000:
        recommendations.append(f"Consider increasing dataset size (current: {total_examples}). 5,000+ examples recommended for good performance.")
    elif total_examples < 5000:
        recommendations.append("Dataset size is adequate but could be larger for better performance.")

    # Mathematical operations recommendations
    math_ops = validation_results["mathematical_operations"]
    for operation, stats in math_ops.items():
        if stats["percentage"] < 15:
            recommendations.append(f"Consider adding more examples with {operation} operations (current: {stats['percentage']:.1f}%)")

    # Reasoning quality recommendations
    reasoning = validation_results["reasoning_analysis"]["quality_indicators"]
    if reasoning["multi_step_reasoning"]["percentage"] < 70:
        recommendations.append("Add more examples with multi-step reasoning for better model training")

    # Source diversity recommendations
    sources = validation_results["source_distribution"]
    if len(sources) == 1:
        recommendations.append("Consider adding examples from multiple data sources for better diversity")

    # Length recommendations
    lengths = validation_results["length_statistics"]
    if lengths["output_lengths"]["avg"] < 200:
        recommendations.append(f"Consider examples with more detailed explanations (current avg output: {lengths['output_lengths']['avg']:.0f} chars)")

    return recommendations

def create_validation_report(validation_results: Dict, output_file: str = None):
    """Create a detailed validation report"""
    if output_file is None:
        output_file = "training/validation_report.json"

    # Save detailed results
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(validation_results, f, indent=2, ensure_ascii=False)

    # Create human-readable summary
    summary_file = output_file.replace('.json', '_summary.txt')

    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write("FINANCIAL REASONING DATA VALIDATION REPORT\n")
        f.write("=" * 50 + "\n\n")

        f.write(f"Dataset Overview:\n")
        f.write(f"- Total Examples: {validation_results['total_examples']}\n")
        f.write(f"- Data Sources: {list(validation_results['source_distribution'].keys())}\n\n")

        f.write("Field Completeness:\n")
        for field, stats in validation_results['field_completeness'].items():
            f.write(f"- {field}: {stats['percentage']:.1f}% complete\n")
        f.write("\n")

        f.write("Mathematical Operations Coverage:\n")
        for op, stats in validation_results['mathematical_operations'].items():
            f.write(f"- {op.title()}: {stats['count']} examples ({stats['percentage']:.1f}%)\n")
        f.write("\n")

        if validation_results['quality_issues']:
            f.write("Quality Issues Identified:\n")
            for issue in validation_results['quality_issues']:
                f.write(f"- {issue}\n")
            f.write("\n")

        if validation_results['recommendations']:
            f.write("Recommendations:\n")
            for rec in validation_results['recommendations']:
                f.write(f"- {rec}\n")
            f.write("\n")

    _dinfo(f"📊 Validation report saved to: {output_file}")
    _dinfo(f"📝 Summary saved to: {summary_file}")

def main():
    """Main validation function"""
    _dinfo("=== PHASE 2: DATA VALIDATION ===")

    # Check if training data exists
    training_file = "training/finqa_clean.json"

    if not os.path.exists(training_file):
        _derr(f"Training data not found: {training_file}")
        _dinfo("Run the data preprocessor cell first")
        return False

    # Run validation
    results = validate_training_data(training_file)

    if "error" in results:
        _derr(f"Validation failed: {results['error']}")
        return False

    # Create report
    create_validation_report(results)

    # Display summary
    quality_score = calculate_quality_score(results)
    reasoning = results["reasoning_analysis"]["quality_indicators"]
    math_ops = results["mathematical_operations"]
    high_coverage = [op for op, stats in math_ops.items() if stats["percentage"] > 30]
    display(Markdown("### Validation summary"))
    display(
        JSON(
            {
                "total_examples": results["total_examples"],
                "quality_score": round(quality_score, 1),
                "reasoning": {
                    "has_calculations_pct": round(reasoning["has_calculations"]["percentage"], 1),
                    "has_final_answer_pct": round(reasoning["has_final_answer"]["percentage"], 1),
                    "multi_step_reasoning_pct": round(reasoning["multi_step_reasoning"]["percentage"], 1),
                },
                "well_covered_operations": high_coverage,
                "quality_issue_count": len(results["quality_issues"]),
                "top_issues": results["quality_issues"][:5],
                "recommendation_count": len(results["recommendations"]),
                "top_recommendations": results["recommendations"][:5],
            }
        )
    )
    if results["quality_issues"]:
        _dwarn(
            f"Quality issues: {len(results['quality_issues'])} (see JSON and validation report on disk)"
        )
    if quality_score >= 80:
        _dinfo("Dataset quality is excellent! Ready for training.")
    elif quality_score >= 60:
        _dinfo("Dataset quality is good. Consider addressing recommendations.")
    else:
        _dwarn("Dataset quality needs improvement. Address issues before training.")
    return quality_score >= 60

def calculate_quality_score(results: Dict) -> float:
    """Calculate an overall quality score (0-100)"""
    score = 100.0

    # Penalize missing fields
    for field in ['instruction', 'input', 'output']:
        missing_pct = 100 - results['field_completeness'][field]['percentage']
        score -= missing_pct * 0.5  # 0.5 points per % missing

    # Penalize quality issues
    quality = results['content_quality']
    score -= quality['malformed_examples']['percentage'] * 2
    score -= max(0, quality['very_short_outputs']['percentage'] - 5) * 0.5

    # Reward good reasoning
    reasoning = results['reasoning_analysis']['quality_indicators']
    if reasoning['has_calculations']['percentage'] < 70:
        score -= (70 - reasoning['has_calculations']['percentage']) * 0.3

    if reasoning['has_final_answer']['percentage'] < 80:
        score -= (80 - reasoning['has_final_answer']['percentage']) * 0.2

    # Ensure score is between 0 and 100
    return max(0.0, min(100.0, score))

if __name__ == "__main__":
    main()
